# Programação dinâmica: o problema da mochila — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula. O código dos algoritmos é escrito em C e compilado com `gcc`
a partir do próprio notebook: cada célula `%%writefile` grava um programa em `src/`, e a célula
seguinte compila e executa. As medições, os gráficos e alguns exercícios ficam em Python.

## Objetivos

Ao final deste tutorial você será capaz de:

- mostrar, com contraexemplos, que o **guloso por valor/peso** resolve a mochila fracionária mas não a **mochila 0/1**;
- escrever a recorrência de $\mathrm{OPT}(i,c)$ e comparar a **recursão direta** (exponencial) com a **memoizada**;
- preencher a tabela $(n+1)\times(W+1)$ **bottom-up** e **reconstruir** uma solução ótima em $O(n)$;
- resolver a **soma de subconjuntos** com uma tabela booleana;
- medir o custo **pseudopolinomial** $\Theta(nW)$ e reduzir o espaço para $\Theta(W)$ com um vetor só.

In [ ]:
# Preparação do ambiente: pasta de trabalho, compilador e funções auxiliares.
import os, subprocess

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

_compilados = {}   # fonte -> data de modificação da última compilação bem-sucedida

def compilar(fonte):
    """Compila src/<fonte> com gcc -O2 (só se o arquivo mudou); devolve o executável ou None."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    mtime = os.path.getmtime(os.path.join('src', fonte))
    if _compilados.get(fonte) == mtime and os.path.exists(exe):
        return exe
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-O2', '-o', exe,
                           os.path.join('src', fonte), '-lm'],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILAÇÃO:\n' + comp.stderr)
        return None
    if comp.stderr:
        print(comp.stderr)
    _compilados[fonte] = mtime
    return exe

def rodar(fonte, args=(), entrada=None, mostrar=True):
    """Compila src/<fonte>, executa com os argumentos e a entrada dados e devolve a saída."""
    exe = compilar(fonte)
    if exe is None:
        return ''
    r = subprocess.run([exe, *map(str, args)], input=entrada, capture_output=True, text=True)
    if mostrar:
        print(r.stdout, end='')
    if r.returncode != 0:
        print(f'(programa terminou com código {r.returncode})', r.stderr)
    return r.stdout

def campo(saida, nome):
    """Extrai o valor de 'nome=valor' da saída dos programas (int se for inteiro, senão float)."""
    for tok in saida.split():
        if tok.startswith(nome + '='):
            valor = tok.split('=')[1]
            return int(valor) if valor.lstrip('-').isdigit() else float(valor)
    raise ValueError(f'{nome} não encontrado em: {saida!r}')

def linha(saida, rotulo):
    """Devolve os inteiros da linha que começa com 'rotulo:' (ex.: 'M3:', 'solucao:')."""
    for lin in saida.splitlines():
        if lin.startswith(rotulo + ':'):
            return [int(x) for x in lin.split(':', 1)[1].split()]
    raise ValueError(f'linha {rotulo} não encontrada em: {saida!r}')

def entrada_mochila(itens, W):
    """Monta a entrada dos programas: 'n W' e depois uma linha 'w v' por item."""
    return f'{len(itens)} {W}\n' + ''.join(f'{w} {v}\n' for w, v in itens)

# Instância condutora dos slides (Levitin, 2012, §8.2): itens (peso, valor), capacidade 5.
ITENS = [(2, 12), (1, 10), (3, 20), (2, 15)]
W_SLIDES = 5
# Instância do guloso (Cormen et al., 2009, §16.2, Figura 16.2): capacidade 50.
ITENS_CLRS = [(10, 60), (20, 100), (30, 120)]

## 1. Mochila 0/1 e o guloso que falha

São $n$ itens com peso $w_i$ e valor $v_i$ e uma mochila de capacidade $W$. Na **mochila 0/1**
cada item entra inteiro ou não entra; na **fracionária** pode-se levar qualquer fração de cada
item. O guloso que ordena por $v_i/w_i$ e vai enchendo a mochila é **ótimo na fracionária**,
mas na 0/1 pode sobrar espaço mal aproveitado: na instância do Cormen ($W=50$) ele leva os
itens 1 e 2 (valor 160), mas o ótimo é levar 2 e 3 (220).

Abaixo, a força bruta (todos os $2^n$ subconjuntos) serve de referência para medir com que
frequência o guloso erra em instâncias aleatórias.

**Fonte:** Cormen et al. (2009), §16.2 e Figura 16.2; Kleinberg e Tardos (2006), §6.4.

In [ ]:
# Exemplo: guloso por valor/peso x força bruta na mochila 0/1.
from itertools import combinations

def forca_bruta(itens, W):
    # testa todos os 2^n subconjuntos e guarda o de maior valor que cabe
    melhor = 0
    for k in range(len(itens) + 1):
        for S in combinations(itens, k):
            if sum(w for w, _ in S) <= W:
                melhor = max(melhor, sum(v for _, v in S))
    return melhor

def guloso_01(itens, W):
    # ordena por valor/peso decrescente e coloca cada item que ainda couber, inteiro
    valor, livre = 0, W
    for w, v in sorted(itens, key=lambda it: it[1] / it[0], reverse=True):
        if w <= livre:
            valor, livre = valor + v, livre - w
    return valor

print('Cormen:  guloso =', guloso_01(ITENS_CLRS, 50), '| ótimo =', forca_bruta(ITENS_CLRS, 50))
print('Slides:  guloso =', guloso_01(ITENS, W_SLIDES), '| ótimo =', forca_bruta(ITENS, W_SLIDES))

In [ ]:
# Visualização: quão longe do ótimo o guloso fica em instâncias aleatórias (n = 12).
import random
import matplotlib.pyplot as plt

random.seed(2026)
razoes = []
for _ in range(400):
    itens = [(random.randint(1, 30), random.randint(1, 50)) for _ in range(12)]
    W = random.randint(20, 120)
    opt = forca_bruta(itens, W)
    if opt > 0:
        razoes.append(guloso_01(itens, W) / opt)

erros = sum(r < 1 for r in razoes)
print(f'O guloso errou em {erros} de {len(razoes)} instâncias; pior razão: {min(razoes):.3f}')
plt.figure(figsize=(6, 3.5))
plt.hist(razoes, bins=30)
plt.xlabel('valor do guloso / valor ótimo')
plt.ylabel('instâncias')
plt.title('Guloso por valor/peso na mochila 0/1')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Exercício 1: a mochila fracionária
# TODO: implemente mochila_fracionaria(itens, W), que devolve o MAIOR valor possível quando
# é permitido levar qualquer fração de cada item (itens = lista de (peso, valor)).
# Use o guloso: ordene por valor/peso decrescente, leve itens inteiros enquanto couberem e,
# no primeiro que não couber, leve só a fração que preenche a mochila.

def mochila_fracionaria(itens, W):
    # TODO: implemente aqui
    pass

# Teste automático
assert mochila_fracionaria(ITENS_CLRS, 50) is not None, 'Implemente mochila_fracionaria()'
assert abs(mochila_fracionaria(ITENS_CLRS, 50) - 240) < 1e-9, 'Cormen, Figura 16.2: 240'
assert abs(mochila_fracionaria(ITENS, W_SLIDES) - 115 / 3) < 1e-9, 'Slides: itens 2 e 4 inteiros + 2/3 do item 3'
assert mochila_fracionaria([(5, 10)], 0) == 0, 'Capacidade zero'
random.seed(7)
for _ in range(200):
    itens = [(random.randint(1, 30), random.randint(1, 50)) for _ in range(10)]
    W = random.randint(0, 120)
    assert mochila_fracionaria(itens, W) >= forca_bruta(itens, W) - 1e-9, \
        'A fracionária nunca vale menos que a 0/1 (é um relaxamento)'
print('Exercício 1 ok!')

## 2. O subproblema $\mathrm{OPT}(i,c)$: recursão direta e memoization

Com um índice só não dá: pôr o item $i$ na mochila não elimina outros itens, mas **reduz a
capacidade**. Por isso o subproblema tem dois índices: $\mathrm{OPT}(i,c)$ é o melhor valor com
os itens $1,\dots,i$ e capacidade $c$:

$$\mathrm{OPT}(0,c) = 0,\qquad
\mathrm{OPT}(i,c) = \begin{cases}\mathrm{OPT}(i-1,c) & \text{se } w_i > c,\\
\max\bigl(\mathrm{OPT}(i-1,c),\ v_i + \mathrm{OPT}(i-1,c-w_i)\bigr) & \text{caso contrário.}\end{cases}$$

A recursão direta faz até $2^{n+1}-1$ chamadas; a memoizada, no máximo $2n(W+1)+1$, pois há só
$n(W+1)$ subproblemas com $i \ge 1$ e cada um faz no máximo duas chamadas. O programa abaixo
gera instâncias com pesos pequenos em relação a $W$, em que quase todo item cabe e a árvore de
recursão fica quase completa.

**Fonte:** Kleinberg e Tardos (2006), §6.4; Cormen et al. (2009), §15.1 (memoization).

In [ ]:
%%writefile src/mochila_rec.c
/* MOCHILA 0/1: recursao direta x memoization (contagem de chamadas).
   Uso: ./mochila_rec n W semente   (instancia aleatoria: pesos 1..W/10, valores 1..100)
   Kleinberg e Tardos (2006), sec. 6.4; Cormen et al. (2009), sec. 15.1. */
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

static int n, W, *w, *v;
static int **T;                 /* T[i][c] == -1: ainda nao calculado */
static long long chamIngenuo = 0, chamMemo = 0;

/* recursao direta: OPT(i,c) = max( OPT(i-1,c), v[i] + OPT(i-1,c-w[i])) */
int optIngenuo( int i, int c) {
   ++chamIngenuo;
   if (i == 0) return 0;
   int r = optIngenuo( i-1, c);
   if (w[i] <= c) {
      int com = v[i] + optIngenuo( i-1, c-w[i]);
      if (com > r) r = com;
   }
   return r;
}

/* a mesma recursao, memoizada */
int optMemo( int i, int c) {
   ++chamMemo;
   if (i == 0) return 0;
   if (T[i][c] >= 0) return T[i][c];
   int r = optMemo( i-1, c);
   if (w[i] <= c) {
      int com = v[i] + optMemo( i-1, c-w[i]);
      if (com > r) r = com;
   }
   return T[i][c] = r;
}

int main( int argc, char *argv[]) {
   if (argc < 4) { fprintf( stderr, "uso: %s n W semente\n", argv[0]); return 1; }
   n = atoi( argv[1]);  W = atoi( argv[2]);  srand( atoi( argv[3]));
   w = malloc( (n+1) * sizeof (int));  v = malloc( (n+1) * sizeof (int));
   for (int i = 1; i <= n; ++i) { w[i] = 1 + rand() % (W/10 > 0 ? W/10 : 1); v[i] = 1 + rand() % 100; }
   T = malloc( (n+1) * sizeof (int *));
   for (int i = 0; i <= n; ++i) {
      T[i] = malloc( (W+1) * sizeof (int));
      for (int c = 0; c <= W; ++c) T[i][c] = -1;
   }
   int a = optMemo( n, W);
   clock_t t0 = clock();
   int b = n <= 30 ? optIngenuo( n, W) : -1;     /* acima disso demora demais */
   double seg = (double) (clock() - t0) / CLOCKS_PER_SEC;
   printf( "n=%d W=%d opt=%d ingenuo=%d chamadas_memo=%lld chamadas_ingenuo=%lld tempo_ingenuo=%.4f\n",
           n, W, a, b, chamMemo, b >= 0 ? chamIngenuo : -1, seg);
   return 0;
}

In [ ]:
# Exemplo: chamadas da recursão direta x memoizada, W = 1000 fixo e n crescendo.
cham_ing, cham_memo, tempo_ing = {}, {}, {}
for n in range(4, 29, 2):
    s = rodar('mochila_rec.c', args=(n, 1000, 1), mostrar=(n % 8 == 4))
    assert campo(s, 'opt') == campo(s, 'ingenuo')
    assert campo(s, 'chamadas_memo') <= 2 * n * 1001 + 1
    cham_ing[n], cham_memo[n] = campo(s, 'chamadas_ingenuo'), campo(s, 'chamadas_memo')
    tempo_ing[n] = campo(s, 'tempo_ingenuo')
print('As duas versões concordam em todas as instâncias.')

In [ ]:
# Visualização: chamadas (escala log). A memoizada é limitada por 2n(W+1)+1.
ns = sorted(cham_ing)
plt.figure(figsize=(6, 4))
plt.semilogy(ns, [cham_ing[n] for n in ns], 'o-', label='recursão direta')
plt.semilogy(ns, [cham_memo[n] for n in ns], 's-', label='memoizada')
plt.semilogy(ns, [2 ** (n + 1) for n in ns], '--', label=r'$2^{n+1}$')
plt.xlabel('n (W = 1000)')
plt.ylabel('chamadas de OPT (escala log)')
plt.title('Mochila: recursão direta x memoization')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.show()

### Exercício 2: soma de subconjuntos

Na **soma de subconjuntos** cada item vale o próprio peso ($v_i = w_i$): quer-se a maior soma
$\le W$. Em vez de valores, a tabela guarda **booleanos**: $A[i][t] = 1$ se algum subconjunto de
$\{1,\dots,i\}$ soma **exatamente** $t$.

$$A[0][0] = 1,\quad A[0][t] = 0\ (t>0),\qquad A[i][t] = A[i-1][t]\ \lor\ \bigl(w_i \le t \land A[i-1][t-w_i]\bigr).$$

A resposta é o maior $t \le W$ com $A[n][t] = 1$. Complete `somaMaxima()`. O `main` compara a sua
resposta com a força bruta em 300 instâncias aleatórias.

In [ ]:
%%writefile src/soma_subconjuntos.c
/* Exercicio 2: SOMA DE SUBCONJUNTOS (o caso v[i] = w[i] da mochila).
   A[i][t] = 1 se algum subconjunto de {1..i} tem soma EXATAMENTE t; 0 caso contrario.
      A[0][0] = 1,  A[0][t] = 0 para t > 0,
      A[i][t] = A[i-1][t]  ou  (w[i] <= t  e  A[i-1][t-w[i]]).
   TODO: complete somaMaxima(), que preenche A bottom-up e devolve a MAIOR soma t <= W
   alcancavel. O main confere a sua resposta com a forca bruta em instancias aleatorias. */
#include <stdio.h>
#include <stdlib.h>

#define NMAX 16
#define WMAX 400

static char A[NMAX+1][WMAX+1];

int somaMaxima( int n, int W, const int w[]) {
   /* TODO: preencha A[0..n][0..W] e devolva o maior t <= W com A[n][t] == 1 */
   return -1;
}

static int forcaBruta( int n, int W, const int w[]) {
   int melhor = 0;
   for (int S = 0; S < (1 << n); ++S) {
      int soma = 0;
      for (int i = 1; i <= n; ++i) if (S & (1 << (i-1))) soma += w[i];
      if (soma <= W && soma > melhor) melhor = soma;
   }
   return melhor;
}

int main( void) {
   int w[NMAX+1];
   srand( 2026);
   for (int teste = 0; teste < 300; ++teste) {
      int n = 1 + rand() % NMAX, W = rand() % (WMAX + 1);
      for (int i = 1; i <= n; ++i) w[i] = 1 + rand() % 60;
      int a = somaMaxima( n, W, w), b = forcaBruta( n, W, w);
      if (a != b) {
         printf( "ERRO: n=%d W=%d, esperado %d, obtido %d\nok=0\n", n, W, b, a);
         return 0;
      }
   }
   int ex[] = {0, 23, 15, 37, 41};   /* nenhum subconjunto soma exatamente 100 */
   printf( "exemplo: W=100 soma_maxima=%d\nok=1\n", somaMaxima( 4, 100, ex));
   return 0;
}

In [ ]:
# Teste automático do Exercício 2
saida = rodar('soma_subconjuntos.c')
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
assert campo(saida, 'soma_maxima') == 93, 'Com 23, 15, 37, 41 e W = 100, a maior soma é 93'
print('Exercício 2 ok!')

## 3. A tabela bottom-up, a reconstrução e as células da memoization

A versão **bottom-up** preenche a matriz $M$ linha por linha: a linha $i$ só depende da linha
$i-1$. Depois, a **reconstrução** refaz as decisões de $(n, W)$ para cima: se
$M[i][c] \ne M[i-1][c]$, o item $i$ entrou e a capacidade cai para $c - w_i$.

O programa abaixo também roda a versão memoizada e conta quantas células com $i \ge 1$ ela de
fato calcula: só as alcançáveis a partir de $(n, W)$. Na instância dos slides são 13 das 24.

**Fonte:** Levitin (2012), §8.2 (mesma instância e "memory functions"); Kleinberg e Tardos (2006), §6.4.

In [ ]:
%%writefile src/mochila.c
/* MOCHILA 0/1 por programacao dinamica: bottom-up, reconstrucao e memoization.
   Uso: ./mochila < entrada     (n W, depois n linhas "w v"; imprime a tabela M,
                                 a solucao, o valor otimo e as celulas da memoization)
   Kleinberg e Tardos (2006), sec. 6.4; Levitin (2012), sec. 8.2. */
#include <stdio.h>
#include <stdlib.h>

static int n, W;
static int *w, *v;              /* w[1..n], v[1..n] */
static int **M;                 /* M[i][c]: melhor valor com itens 1..i e capacidade c */
static int **T;                 /* tabela da memoization; T[i][c] == -1: nao calculado */
static long long celulasMemo = 0;

static int **matriz( int lin, int col, int valor) {
   int **A = malloc( lin * sizeof (int *));
   for (int i = 0; i < lin; ++i) {
      A[i] = malloc( col * sizeof (int));
      for (int j = 0; j < col; ++j) A[i][j] = valor;
   }
   return A;
}

/* bottom-up: a linha i depende so da linha i-1 */
int mochila( void) {
   for (int c = 0; c <= W; ++c) M[0][c] = 0;
   for (int i = 1; i <= n; ++i)
      for (int c = 0; c <= W; ++c) {
         M[i][c] = M[i-1][c];                          /* i fora   */
         if (w[i] <= c && v[i] + M[i-1][c-w[i]] > M[i][c])
            M[i][c] = v[i] + M[i-1][c-w[i]];           /* i dentro */
      }
   return M[n][W];
}

/* refaz as decisoes de baixo para cima; imprime os itens em ordem crescente */
void solucao( void) {
   int *escolhido = calloc( n + 1, sizeof (int));
   int c = W;
   for (int i = n; i >= 1; --i)
      if (M[i][c] != M[i-1][c]) {                      /* o item i entrou */
         escolhido[i] = 1;
         c -= w[i];
      }
   for (int i = 1; i <= n; ++i) if (escolhido[i]) printf( " %d", i);
   free( escolhido);
}

/* a mesma recorrencia, de cima para baixo, com memoization */
int memo( int i, int c) {
   if (i == 0) return 0;
   if (T[i][c] < 0) {
      ++celulasMemo;
      int r = memo( i-1, c);
      if (w[i] <= c && v[i] + memo( i-1, c-w[i]) > r)
         r = v[i] + memo( i-1, c-w[i]);
      T[i][c] = r;
   }
   return T[i][c];
}

int main( void) {
   if (scanf( "%d %d", &n, &W) != 2) return 1;
   w = malloc( (n+1) * sizeof (int));  v = malloc( (n+1) * sizeof (int));
   for (int i = 1; i <= n; ++i)
      if (scanf( "%d %d", &w[i], &v[i]) != 2) return 1;
   M = matriz( n+1, W+1, 0);
   T = matriz( n+1, W+1, -1);
   int opt = mochila();
   int optMemo = memo( n, W);
   if (W <= 40)
      for (int i = 0; i <= n; ++i) {
         printf( "M%d:", i);
         for (int c = 0; c <= W; ++c) printf( " %d", M[i][c]);
         printf( "\n");
      }
   printf( "solucao:");  solucao();  printf( "\n");
   printf( "opt=%d memo=%d celulas_memo=%lld celulas_tabela=%lld\n",
           opt, optMemo, celulasMemo, (long long) n * (W+1));
   return 0;
}

In [ ]:
# Exemplo: a instância dos slides.
saida = rodar('mochila.c', entrada=entrada_mochila(ITENS, W_SLIDES))
assert linha(saida, 'M3') == [0, 10, 12, 22, 30, 32]
assert linha(saida, 'M4') == [0, 10, 15, 25, 30, 37]
assert linha(saida, 'solucao') == [1, 2, 4] and campo(saida, 'opt') == 37
assert campo(saida, 'celulas_memo') == 13
print('Instância dos slides conferida: solução {1, 2, 4}, valor 37, 13 células memoizadas.')

In [ ]:
# Visualização: a tabela M como mapa de calor, com o caminho da reconstrução.
import numpy as np

M = np.array([linha(saida, f'M{i}') for i in range(len(ITENS) + 1)])
caminho, c = [], W_SLIDES
for i in range(len(ITENS), 0, -1):
    caminho.append((i, c))
    if M[i][c] != M[i - 1][c]:
        c -= ITENS[i - 1][0]
caminho.append((0, c))

plt.figure(figsize=(6, 3.8))
plt.imshow(M, cmap='Blues')
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        plt.text(j, i, M[i, j], ha='center', va='center',
                 color='white' if M[i, j] > 25 else 'black')
ys, xs = zip(*caminho)
plt.plot(xs, ys, 'o-', color='tab:orange', lw=2, ms=22, mfc='none', mew=2)
plt.xticks(range(W_SLIDES + 1), [f'c={c}' for c in range(W_SLIDES + 1)])
plt.yticks(range(len(ITENS) + 1),
           ['i=0'] + [f'{i} {ITENS[i-1]}' for i in range(1, len(ITENS) + 1)])
plt.title('Tabela M dos slides (em laranja: a reconstrução)')
plt.show()

### Exercício 3: reconstruindo uma solução ótima

A função `tabela()` abaixo já preenche $M$. Complete `reconstroi()`: ela deve gravar em
`escolhidos[]` os itens de **uma** solução ótima e devolver quantos são, usando só a tabela e os
pesos, em $O(n)$ passos. O `main` confere, em 400 instâncias aleatórias, que os itens não se
repetem, que cabem na mochila e que o valor é o da força bruta.

In [ ]:
%%writefile src/reconstroi.c
/* Exercicio 3: RECONSTRUINDO A SOLUCAO a partir da tabela M.
   tabela() preenche M[0..n][0..W] como nos slides (ja esta pronta).
   TODO: complete reconstroi(): grave em escolhidos[] os itens de UMA solucao otima, em
   qualquer ordem, e devolva quantos sao. Use so a tabela M e os pesos w[]: O(n) passos.
   O main confere peso, valor e repeticoes contra a forca bruta em instancias aleatorias. */
#include <stdio.h>
#include <stdlib.h>

#define NMAX 14
#define WMAX 120

static int M[NMAX+1][WMAX+1];

void tabela( int n, int W, const int w[], const int v[]) {
   for (int c = 0; c <= W; ++c) M[0][c] = 0;
   for (int i = 1; i <= n; ++i)
      for (int c = 0; c <= W; ++c) {
         M[i][c] = M[i-1][c];
         if (w[i] <= c && v[i] + M[i-1][c-w[i]] > M[i][c])
            M[i][c] = v[i] + M[i-1][c-w[i]];
      }
}

int reconstroi( int n, int W, const int w[], int escolhidos[]) {
   int k = 0;
   /* TODO: percorra i = n, n-1, ..., 1 decidindo, pela tabela, se o item i entrou */
   return k;
}

static int forcaBruta( int n, int W, const int w[], const int v[]) {
   int melhor = 0;
   for (int S = 0; S < (1 << n); ++S) {
      int peso = 0, valor = 0;
      for (int i = 1; i <= n; ++i)
         if (S & (1 << (i-1))) { peso += w[i]; valor += v[i]; }
      if (peso <= W && valor > melhor) melhor = valor;
   }
   return melhor;
}

int main( void) {
   int w[NMAX+1], v[NMAX+1], esc[NMAX+1], usado[NMAX+1];
   int wl[] = {0, 2, 1, 3, 2}, vl[] = {0, 12, 10, 20, 15};      /* instancia dos slides */
   tabela( 4, 5, wl, vl);
   int k = reconstroi( 4, 5, wl, esc);
   printf( "slides:");  for (int j = 0; j < k; ++j) printf( " %d", esc[j]);  printf( "\n");
   srand( 2026);
   for (int teste = 0; teste < 400; ++teste) {
      int n = 1 + rand() % NMAX, W = rand() % (WMAX + 1);
      for (int i = 1; i <= n; ++i) { w[i] = 1 + rand() % 40; v[i] = 1 + rand() % 50; }
      tabela( n, W, w, v);
      k = reconstroi( n, W, w, esc);
      int peso = 0, valor = 0;
      for (int i = 0; i <= n; ++i) usado[i] = 0;
      for (int j = 0; j < k; ++j) {
         if (esc[j] < 1 || esc[j] > n || usado[esc[j]]) {
            printf( "ERRO: item invalido ou repetido (%d)\nok=0\n", esc[j]);  return 0;
         }
         usado[esc[j]] = 1;  peso += w[esc[j]];  valor += v[esc[j]];
      }
      int opt = forcaBruta( n, W, w, v);
      if (peso > W || valor != opt) {
         printf( "ERRO: n=%d W=%d: peso %d, valor %d; o otimo vale %d\nok=0\n", n, W, peso, valor, opt);
         return 0;
      }
   }
   printf( "ok=1\n");
   return 0;
}

In [ ]:
# Teste automático do Exercício 3
saida = rodar('reconstroi.c')
assert sorted(linha(saida, 'slides')) == [1, 2, 4], 'Nos slides a solução é {1, 2, 4}'
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
print('Exercício 3 ok!')

## 4. Custo pseudopolinomial e economia de espaço

A tabela tem $(n+1)(W+1)$ células, então o tempo é $\Theta(nW)$: linear no **valor** de $W$.
Mas $W$ ocupa só $k \approx \log_2 W$ bits na entrada; com $W \approx 2^k$, o tempo é
$\Theta(n\,2^k)$, **exponencial** no tamanho de $W$. Algoritmos assim são **pseudopolinomiais**.

O programa abaixo guarda só **duas linhas** da tabela (a anterior e a atual), o que basta para
o valor ótimo e reduz o espaço a $\Theta(W)$. Com $n = 100$ fixo, medimos o tempo dobrando $W$:
cada bit a mais em $W$ dobra o tempo.

**Fonte:** Kleinberg e Tardos (2006), §6.4; Garey e Johnson (1979), §4.2.

In [ ]:
%%writefile src/mochila_tempo.c
/* MOCHILA 0/1 bottom-up com DUAS linhas (espaco O(W)), para medir o tempo em funcao de W.
   Uso: ./mochila_tempo n W semente   (pesos 1..W/2, valores 1..1000)
   Kleinberg e Tardos (2006), sec. 6.4. */
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

int main( int argc, char *argv[]) {
   if (argc < 4) { fprintf( stderr, "uso: %s n W semente\n", argv[0]); return 1; }
   int n = atoi( argv[1]), W = atoi( argv[2]);
   srand( atoi( argv[3]));
   int *w = malloc( (n+1) * sizeof (int)), *v = malloc( (n+1) * sizeof (int));
   for (int i = 1; i <= n; ++i) { w[i] = 1 + rand() % (W/2 > 0 ? W/2 : 1); v[i] = 1 + rand() % 1000; }
   long long *ant = calloc( W+1, sizeof (long long));   /* linha i-1 */
   long long *atu = calloc( W+1, sizeof (long long));   /* linha i   */
   clock_t t0 = clock();
   for (int i = 1; i <= n; ++i) {
      for (int c = 0; c <= W; ++c) {
         atu[c] = ant[c];
         if (w[i] <= c && v[i] + ant[c-w[i]] > atu[c]) atu[c] = v[i] + ant[c-w[i]];
      }
      long long *tmp = ant;  ant = atu;  atu = tmp;      /* a linha i vira a "anterior" */
   }
   double seg = (double) (clock() - t0) / CLOCKS_PER_SEC;
   printf( "n=%d W=%d opt=%lld tempo=%.4f\n", n, W, ant[W], seg);
   return 0;
}

In [ ]:
# Exemplo: tempo da PD com n = 100 e W = 2^k, k = 12..23.
tempos = {}
for k in range(12, 24):
    s = rodar('mochila_tempo.c', args=(100, 2 ** k, 1), mostrar=(k % 4 == 3))
    tempos[k] = campo(s, 'tempo')
print(f'k = 23 levou {tempos[23] / max(tempos[20], 1e-6):.1f} vezes o tempo de k = 20 (esperado: ~8).')

In [ ]:
# Visualização: tempo em função de W (linear) e do número de bits k (exponencial).
ks = sorted(tempos)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
a1.plot([2 ** k for k in ks], [tempos[k] for k in ks], 'o-')
a1.set_xlabel('W')
a1.set_ylabel('tempo (s)')
a1.set_title('Linear no valor de W')
a1.grid(True, alpha=0.3)
a2.semilogy(ks, [tempos[k] for k in ks], 's-', color='tab:red')
a2.set_xlabel('k = bits de W')
a2.set_ylabel('tempo (s, escala log)')
a2.set_title('Exponencial no tamanho de W')
a2.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

### Exercício 4: um vetor só

Dá para usar **um** vetor `m[0..W]` em vez de duas linhas: para cada item $i$, faça
`m[c] = max(m[c], v[i] + m[c - w[i]])` para os $c \ge w_i$. O detalhe é o **sentido** do laço em
$c$: quando `m[c - w[i]]` é lido, ele ainda precisa conter o valor da linha $i-1$. Complete
`mochila1D()`. O `main` compara com a versão de duas linhas em 500 instâncias aleatórias.

Depois que passar, troque o sentido do laço e rode de novo: que valor sai na instância dos
slides, e por quê? (A resposta prepara o desafio final.)

In [ ]:
%%writefile src/mochila1d.c
/* Exercicio 4: MOCHILA 0/1 COM UM VETOR SO (espaco O(W)).
   m[c] comeca com 0 (linha i = 0). Para cada item i, atualize m[] de forma que, ao final,
   m[c] seja a linha i da tabela: m[c] = max( m[c], v[i] + m[c-w[i]]) quando w[i] <= c.
   TODO: complete mochila1D(). Atencao ao SENTIDO do laco em c (veja os slides).
   O main compara com a versao de duas linhas em instancias aleatorias. */
#include <stdio.h>
#include <stdlib.h>

long long mochila1D( int n, int W, const int w[], const int v[]) {
   long long *m = calloc( W+1, sizeof (long long));
   /* TODO: para i = 1..n, atualize m[] */
   long long r = m[W];
   free( m);
   return r;
}

static long long duasLinhas( int n, int W, const int w[], const int v[]) {
   long long *ant = calloc( W+1, sizeof (long long)), *atu = calloc( W+1, sizeof (long long));
   for (int i = 1; i <= n; ++i) {
      for (int c = 0; c <= W; ++c) {
         atu[c] = ant[c];
         if (w[i] <= c && v[i] + ant[c-w[i]] > atu[c]) atu[c] = v[i] + ant[c-w[i]];
      }
      long long *tmp = ant;  ant = atu;  atu = tmp;
   }
   long long r = ant[W];
   free( ant);  free( atu);
   return r;
}

int main( void) {
   int w[61], v[61];
   int wl[] = {0, 2, 1, 3, 2}, vl[] = {0, 12, 10, 20, 15};
   printf( "slides=%lld\n", mochila1D( 4, 5, wl, vl));
   srand( 2026);
   for (int teste = 0; teste < 500; ++teste) {
      int n = 1 + rand() % 60, W = rand() % 2000;
      for (int i = 1; i <= n; ++i) { w[i] = 1 + rand() % 300; v[i] = 1 + rand() % 1000; }
      long long a = mochila1D( n, W, w, v), b = duasLinhas( n, W, w, v);
      if (a != b) {
         printf( "ERRO: n=%d W=%d: esperado %lld, obtido %lld\nok=0\n", n, W, b, a);
         return 0;
      }
   }
   printf( "ok=1\n");
   return 0;
}

In [ ]:
# Teste automático do Exercício 4
saida = rodar('mochila1d.c')
assert campo(saida, 'slides') == 37, 'Na instância dos slides o ótimo é 37'
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
print('Exercício 4 ok!')

## Desafio Final: a mochila com repetição

Agora há um **estoque ilimitado** de cada item: o item $i$ pode entrar $0, 1, 2, \dots$ vezes.
Exemplo (Dasgupta et al., 2008, §6.4): com $W = 10$ e itens (peso, valor)
$(6, 30), (3, 14), (4, 16), (2, 9)$, o ótimo **sem** repetição vale $46$ (itens 1 e 3); **com**
repetição vale $48$ (o item 1 e duas cópias do item 4).

1. **Recorrência com um índice.** Seja $K(c)$ o melhor valor com capacidade $c$. Olhando para
   *um item qualquer* da solução ótima, escreva a recorrência de $K(c)$ e prove a subestrutura
   ótima pelo argumento recorta e cola. Por que aqui **não** é preciso o índice $i$?
2. **Implementação.** Escreva `src/mochila_rep.c` em $\Theta(nW)$ tempo e $\Theta(W)$ espaço,
   com a reconstrução (quantas cópias de cada item). Confira $K(10) = 48$.
3. **Ligação com o Exercício 4.** Mostre que o laço de `mochila1D()` com $c$ **crescente**
   resolve exatamente este problema. Use `src/mochila.c` para confirmar o $46$ da versão 0/1.
4. **Pseudopolinomial de novo.** O tempo continua $\Theta(nW)$. Construa uma família de
   instâncias com $n = 2$ em que o número de bits da entrada cresce como $k$ e o tempo como $2^k$.

In [ ]:
# Desafio: a mochila com repetição
# TODO: resolva o desafio abaixo
# Sugestão: confira primeiro a versão 0/1 da instância de Dasgupta:
#   rodar('mochila.c', entrada=entrada_mochila([(6, 30), (3, 14), (4, 16), (2, 9)], 10))

# Sua solução aqui

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. As principais para este tutorial:

- KLEINBERG, J.; TARDOS, É. *Algorithm Design*. Pearson/Addison-Wesley, 2006. §6.4.
- LEVITIN, A. *Introduction to the Design and Analysis of Algorithms*. 3. ed. Pearson, 2012. §8.2.
- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. §16.2 e §34.5.5. (Em português: *Algoritmos: Teoria e Prática*, Elsevier, 2012.)
- DASGUPTA, S.; PAPADIMITRIOU, C. H.; VAZIRANI, U. *Algorithms*. McGraw-Hill, 2008. §6.4.
- GAREY, M. R.; JOHNSON, D. S. *Computers and Intractability*. W. H. Freeman, 1979. §4.2.
- FEOFILOFF, P. *Mochila booleana*. Análise de Algoritmos, IME-USP. https://www.ime.usp.br/~pf/analise_de_algoritmos/aulas/mochila-bool.html